# Phase 6: Optuna Tuning
Hyperparameter tuning for XGBoost.

In [ ]:
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../data/processed/train_sample_processed.csv')
y = df['isFraud']
X = df.drop(columns=['isFraud'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

def objective(trial):
    params = {
        'n_estimators': 200, # lower for speed in tuning
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 100, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 100, log=True),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': 'auc'
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False, early_stopping_rounds=20)
    
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # 10 for demo, 30-50 for real

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)
